# Fake News Detection Model

This notebook trains a Logistic Regression model with TF-IDF vectorization on the dataset.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import seaborn as sns
import re
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, log_loss

## 1. Load Data
Load the pre-split train and test datasets.

In [ ]:
print("Loading dataset...")
train_df = pd.read_csv('../dataset/final_dataset/train_split.csv')
test_df = pd.read_csv('../dataset/final_dataset/test_split.csv')

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/final_dataset/train_split.csv'

## 2. Preprocess Data
Handle any missing values in the text column.

In [ ]:
# Fill missing text with empty string just in case
train_df['text'] = train_df['text'].fillna('')
test_df['text'] = test_df['text'].fillna('')

X_train = train_df['text']
y_train = train_df['label']

X_test = test_df['text']
y_test = test_df['label']

## 3. Feature Extraction (TF-IDF)
Convert text into numerical features using TF-IDF Vectorizer.

In [ ]:
print("Building text pipeline...")

class TextStructureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for text in X:
            text_str = str(text)
            total_chars = len(text_str) if len(text_str) > 0 else 1
            words = text_str.split()
            total_words = len(words) if len(words) > 0 else 1
            exclamation_freq = text_str.count('!') / total_chars
            question_freq = text_str.count('?') / total_chars
            uppercase_words = sum(1 for w in words if w.isupper() and len(re.sub(r'[^a-zA-Z]', '', w)) > 1)
            uppercase_freq = uppercase_words / total_words
            features.append([exclamation_freq, question_freq, uppercase_freq])
        return np.asarray(features, dtype=float)

feature_union = FeatureUnion([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 3), stop_words='english', max_features=50000)),
    ('structure', TextStructureExtractor())
])

pipeline = Pipeline([
    ('features', feature_union),
    ('clf', SGDClassifier(loss='log_loss', learning_rate='constant', eta0=0.01, random_state=42))
])

X_train_features = pipeline.named_steps['features'].fit_transform(X_train)
X_test_features = pipeline.named_steps['features'].transform(X_test)
vectorizer = pipeline.named_steps['features'].transformer_list[0][1]

print(f"Feature matrix shape (train): {X_train_features.shape}")

## 4. Model Training (Logistic Regression)
Train a Logistic Regression model on the TF-IDF features.

In [ ]:
print("Training classifier...")
epochs = 30
classes = np.unique(y_train)
clf = pipeline.named_steps['clf']
train_losses = []
hit1_scores = []
hit10_scores = []
class_to_index = {label: idx for idx, label in enumerate(classes)}

def topk_accuracy(y_true, y_prob, k):
    k = min(k, y_prob.shape[1])
    true_indices = np.array([class_to_index[label] for label in y_true])
    topk_indices = np.argsort(y_prob, axis=1)[:, -k:]
    return np.mean([true_indices[i] in topk_indices[i] for i in range(len(true_indices))])

n_samples = X_train_features.shape[0]
for epoch in range(1, epochs + 1):
    shuffled_idx = np.random.RandomState(42 + epoch).permutation(n_samples)
    X_epoch = X_train_features[shuffled_idx]
    y_epoch = y_train.iloc[shuffled_idx].reset_index(drop=True)
    clf.partial_fit(X_epoch, y_epoch, classes=classes)
    y_train_prob = clf.predict_proba(X_train_features)
    train_losses.append(log_loss(y_train, y_train_prob))
    y_test_prob = clf.predict_proba(X_test_features)
    hit1_scores.append(topk_accuracy(y_test, y_test_prob, 1))
    hit10_scores.append(topk_accuracy(y_test, y_test_prob, 10))
    print(f"Epoch {epoch}/{epochs} - loss: {train_losses[-1]:.4f} - hits@1: {hit1_scores[-1]:.4f} - hits@10: {hit10_scores[-1]:.4f}")

pipeline.steps[-1] = ('clf', clf)
model = pipeline

## 5. Evaluation
Evaluate the model on the test dataset.

In [ ]:
print("Evaluating model...")
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fake', 'Real'])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

## 6. Research Paper Visualizations
Generate high-quality plots suitable for a research paper: ROC Curve, Precision-Recall Curve, and Feature Importance.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 300})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(range(1, epochs + 1), train_losses, marker='o', color='crimson', linewidth=2)
ax1.set_title('Training Loss Convergence')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Log Loss')

ax2.plot(range(1, epochs + 1), hit1_scores, marker='o', color='teal', linewidth=2)
ax2.set_title('Evaluation Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 300})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

rdisp = RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax1)
ax1.set_title('Receiver Operating Characteristic (ROC)')
ax1.plot([0, 1], [0, 1], linestyle='--', color='navy')
try:
    rdisp.line_.set_color('darkorange')
except Exception:
    try:
        ax1.lines[-1].set_color('darkorange')
    except Exception:
        pass

prdisp = PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax2)
ax2.set_title('Precision-Recall Curve')
try:
    prdisp.line_.set_color('green')
except Exception:
    try:
        ax2.lines[-1].set_color('green')
    except Exception:
        pass

plt.tight_layout()
plt.show()

In [ ]:
tfidf_feature_names = model.named_steps['features'].transformer_list[0][1].get_feature_names_out()
structural_feature_names = np.array(['exclamation_mark_frequency', 'question_mark_frequency', 'uppercase_word_frequency'])
feature_names = np.concatenate([tfidf_feature_names, structural_feature_names])
coefs = model.named_steps['clf'].coef_[0]

coef_df = pd.DataFrame({'feature': feature_names, 'coef': coefs})
top_real = coef_df.sort_values(by='coef', ascending=False).head(20)
top_fake = coef_df.sort_values(by='coef', ascending=True).head(20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

colors_real = sns.color_palette('Blues_r', n_colors=len(top_real))
ax1.barh(top_real['feature'], top_real['coef'], color=colors_real)
ax1.invert_yaxis()
ax1.set_title('Top 20 Features indicating Real News')
ax1.set_xlabel('Coefficient')
ax1.set_ylabel('')

colors_fake = sns.color_palette('Reds_r', n_colors=len(top_fake))
ax2.barh(top_fake['feature'], top_fake['coef'], color=colors_fake)
ax2.invert_yaxis()
ax2.set_title('Top 20 Features indicating Fake News')
ax2.set_xlabel('Coefficient')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
sample_texts = [
    "Donald Trump is an Indian",
    "Rahul Gandhi is the PM of India",
    "The Earth is flat and NASA is hiding it"
]

predictions = model.predict(sample_texts)
probabilities = model.predict_proba(sample_texts)
class_names = {0: 'Fake', 1: 'Real'}

for i, text in enumerate(sample_texts):
    pred_class = int(predictions[i])
    confidence = float(np.max(probabilities[i]) * 100)
    print(f"Sample {i + 1}: {text}")
    print(f"Prediction: {class_names[pred_class]}")
    print(f"Confidence: {confidence:.2f}%")
    print('-' * 40)